<a href="https://colab.research.google.com/github/Sowmyaindugu1505/Driver-Drowsiness-Detection-using-Deep-learning/blob/main/Driver_Drowsiness_Detection/Driver_Drowsiness_Detection/Driver_Drowsiness_Detection_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Driver Drowsiness Detection (Eye + Yawn) - Google Colab Notebook

This notebook runs the complete workflow end-to-end:
1. Setup dependencies
2. Load `archive.zip` dataset
3. Prepare Eye + Yawn train/valid/test splits
4. Train eye model
5. Evaluate eye model
6. Train yawn model
7. Evaluate yawn model
8. Save artifacts

> **Important:** Live webcam alert (`detect.py`) is best run on your local laptop/desktop, not in Colab.

## 0) Runtime recommendation
- Colab menu -> **Runtime -> Change runtime type -> GPU** (recommended).
- If you use CPU, training still works but will be slower.

In [7]:
!python --version
!nvidia-smi || true

Python 3.12.12
Sun Mar  8 08:45:17 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   42C    P8             14W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+--------------------------------

## 1) Clone repository and move to project folder

In [8]:
# If already cloned, this will skip cloning and just use existing folder
import os
from pathlib import Path

REPO_URL = 'https://github.com/<sowmya>/Driver-Drowsiness-Detection-using-Deep-learning.git'
REPO_DIR = Path('/content/Driver-Drowsiness-Detection-using-Deep-learning')

if not REPO_DIR.exists():
    !git clone {REPO_URL}

PROJECT_DIR = REPO_DIR / 'Driver_Drowsiness_Detection' / 'Driver_Drowsiness_Detection'
print('PROJECT_DIR =', PROJECT_DIR)
assert PROJECT_DIR.exists(), f'Project folder not found: {PROJECT_DIR}'
%cd $PROJECT_DIR

/bin/bash: line 1: sowmya: No such file or directory
PROJECT_DIR = /content/Driver-Drowsiness-Detection-using-Deep-learning/Driver_Drowsiness_Detection/Driver_Drowsiness_Detection


AssertionError: Project folder not found: /content/Driver-Drowsiness-Detection-using-Deep-learning/Driver_Drowsiness_Detection/Driver_Drowsiness_Detection

## 2) Install dependencies

In [ ]:
!pip -q install tensorflow==2.15.0 opencv-python-headless==4.10.0.84 pygame==2.6.1 scikit-learn

## 3) Upload `archive.zip`
Upload the dataset zip from your computer if it is not already present.

In [ ]:
from pathlib import Path
from google.colab import files

archive_path = Path('archive.zip')
if not archive_path.exists():
    print('Please upload archive.zip now...')
    uploaded = files.upload()
    if 'archive.zip' not in uploaded:
        raise FileNotFoundError('Upload must include a file named archive.zip')

assert archive_path.exists(), 'archive.zip not found after upload'
print('archive.zip ready at', archive_path.resolve())

## 4) Prepare combined eye+yawn dataset from zip

In [ ]:
!python prepare_combined_dataset.py

In [ ]:
from pathlib import Path

def count_images(folder):
    exts = {'.png','.jpg','.jpeg','.bmp','.webp'}
    p = Path(folder)
    return sum(1 for f in p.rglob('*') if f.is_file() and f.suffix.lower() in exts)

for split in ['train','valid','test']:
    for cls in ['open','closed']:
        print(f'data/{split}/{cls}:', count_images(Path('data')/split/cls))

for split in ['train','valid','test']:
    for cls in ['yawn','no_yawn']:
        print(f'data_yawn/{split}/{cls}:', count_images(Path('data_yawn')/split/cls))

## 5) Train eye model

In [ ]:
!python train.py

## 6) Evaluate eye model

In [ ]:
!python evaluate_eye.py

## 7) Train yawn model

In [ ]:
!python train_yawn.py

## 8) Evaluate yawn model

In [ ]:
!python evaluate_yawn.py

## 9) Verify output artifacts
After successful run you should see:
- `models/cnnCat2.h5` (eye model)
- `models/yawn_cnn.h5` (yawn model)

In [ ]:
from pathlib import Path
for model_file in ['models/cnnCat2.h5','models/yawn_cnn.h5']:
    p = Path(model_file)
    print(model_file, '->', 'FOUND' if p.exists() else 'MISSING')

## 10) (Optional) Save artifacts to Google Drive
Uncomment and run if you want models/checkpoints saved permanently.

In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')
# !mkdir -p /content/drive/MyDrive/drowsiness_models
# !cp -v models/cnnCat2.h5 models/yawn_cnn.h5 /content/drive/MyDrive/drowsiness_models/

## 11) Run live detector
Run this on your local machine (recommended), because Colab webcam and `pygame` alarm are not reliable for real-time app usage.

```bash
python detect.py
```
Press **q** to stop.